# 1. Setup & Imports
This section imports required libraries, configures paths, and defines the experiment scope for FinSight XGBoost forecasting.

In [1]:
import sys
sys.path.append("../../../")

import json
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

from backend.data.data_loader import fetch_stock_data
from backend.data.preprocessor import FinSightPreprocessor

optuna.logging.set_verbosity(optuna.logging.INFO)

TICKER = "RELIANCE.NS"
START = "2020-01-01"
END = "2026-05-01"
TEST_START = "2025-01-01"

safe_ticker = ''.join(ch if ch.isalnum() else '_' for ch in TICKER)
ARTIFACTS_DIR = Path("./")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker: {TICKER}")
print(f"Date range: {START} to {END}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")

c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ticker: RELIANCE.NS
Date range: 2020-01-01 to 2026-05-01
Artifacts dir: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost


# 2. Load Data
Load OHLCV data using FinSight's shared data loader and inspect shape and schema.

In [2]:
df = fetch_stock_data(TICKER, START, END)
print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (1567, 9)


Price,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread
Date,,,,,,,,,
2020-01-02,691.235535,704.470520,691.235535,701.887512,17710316,686.821228,0.017024,0.016881,13.234985
2020-01-03,700.835999,704.790527,696.264343,702.733276,20984698,687.648865,0.001205,0.001204,8.526184
2020-01-06,694.892883,698.504456,684.835205,686.435303,24519177,671.700684,-0.023192,-0.023465,13.669250
2020-01-07,694.435669,701.521790,691.921265,696.995850,16683622,682.034546,0.015385,0.015267,9.600525
2020-01-08,692.607056,701.498901,690.321228,691.761292,16047902,676.912354,-0.007510,-0.007539,11.177673


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1567 entries, 2020-01-02 to 2026-05-01
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1567 non-null   float64
 1   High          1567 non-null   float64
 2   Low           1567 non-null   float64
 3   Close         1567 non-null   float64
 4   Volume        1567 non-null   int64  
 5   Adj Close     1567 non-null   float64
 6   daily_return  1567 non-null   float64
 7   log_return    1567 non-null   float64
 8   hl_spread     1567 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 122.4 KB


# 3. Feature Engineering
Apply FinSight preprocessing and optionally enrich the feature frame using graph features if the JSON artifact exists.
The target variable is next-day Close (`Close.shift(-1)`).

In [3]:
preprocessor = FinSightPreprocessor(ticker=TICKER)
processed_df = preprocessor.fit_transform(df)

graph_features_path = ARTIFACTS_DIR / "graph_features.json"
if graph_features_path.exists():
    with graph_features_path.open("r", encoding="utf-8") as f:
        graph_features = json.load(f)
    for k, v in graph_features.items():
        processed_df[k] = float(v)
    print(f"Loaded graph features from {graph_features_path}")
else:
    graph_features = {}
    print(f"No graph features found at {graph_features_path}; continuing without them.")

model_df = processed_df.copy()
model_df["target"] = model_df["Close"].shift(-1)
model_df = model_df.dropna(subset=["target"]).replace([np.inf, -np.inf], np.nan).dropna()

print("Final model frame shape:", model_df.shape)
display(model_df.head())
print("Feature columns (excluding target):", len([c for c in model_df.columns if c != "target"]))

No graph features found at graph_features.json; continuing without them.
Final model frame shape: (1547, 15)


,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread,Close_lag_1,Close_lag_5,Close_lag_10,rolling_mean_20,rolling_std_20,target
Date,,,,,,,,,,,,,,,
2020-01-29,-2.163245,-2.171218,-2.131566,-2.147750,0.447922,-2.132590,0.273794,0.282282,-0.783040,-2.159071,-2.022229,-2.020639,-2.036570,-1.426146,-2.218431
2020-01-30,-2.153547,-2.200000,-2.178682,-2.218431,0.291245,-2.200770,-1.410867,-1.421472,-0.432743,-2.143230,-2.034888,-1.993429,-2.045523,-1.217701,-2.281280
2020-01-31,-2.204485,-2.251787,-2.242940,-2.281280,1.116623,-2.261395,-1.289124,-1.296610,-0.194839,-2.213831,-2.045209,-1.909956,-2.057796,-0.929529,-2.332479
2020-02-03,-2.367291,-2.356144,-2.329434,-2.332479,0.846701,-2.310783,-1.080113,-1.082887,-0.537643,-2.276609,-2.074421,-2.004177,-2.069140,-0.600548,-2.252400
2020-02-04,-2.308320,-2.292414,-2.261356,-2.252400,0.490524,-2.233537,1.627040,1.614568,-0.620069,-2.327750,-2.142193,-2.001175,-2.078743,-0.488965,-2.209131


Feature columns (excluding target): 14


# 4. Train/Test Split
Create a chronological split with the last 20% as test data (no shuffling).

In [4]:
train_df = model_df[model_df.index < "2025-01-01"]
test_df = model_df[model_df.index >= "2025-01-01"]

feature_cols = [c for c in model_df.columns if c != "target"]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test = test_df[feature_cols], test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Features:", len(feature_cols))

Train size: 1218
Test size: 329
Features: 14


# 5. Time Series Cross Validation
Run baseline cross-validation using `TimeSeriesSplit(n_splits=5)` and report fold RMSE.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)
baseline_rmses = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror",
        tree_method="hist",
        device="cuda"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

    preds = model.predict(X_val)
    rmse = float(np.sqrt(mean_squared_error(y_val, preds)))
    baseline_rmses.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.6f}")

print(f"Mean CV RMSE: {np.mean(baseline_rmses):.6f}")

[0]	validation_0-rmse:0.66931
[100]	validation_0-rmse:0.10669
[200]	validation_0-rmse:0.10496
[300]	validation_0-rmse:0.10511
[400]	validation_0-rmse:0.10526
[499]	validation_0-rmse:0.10525
Fold 1 RMSE: 0.105249
[0]	validation_0-rmse:1.19049


c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\xgboost\core.py:751: UserWarning: [19:43:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[100]	validation_0-rmse:0.36328
[200]	validation_0-rmse:0.35542
[300]	validation_0-rmse:0.35508
[400]	validation_0-rmse:0.35522
[499]	validation_0-rmse:0.35510
Fold 2 RMSE: 0.355096
[0]	validation_0-rmse:0.73280
[100]	validation_0-rmse:0.07303
[200]	validation_0-rmse:0.07568
[300]	validation_0-rmse:0.07676
[400]	validation_0-rmse:0.07715
[499]	validation_0-rmse:0.07737
Fold 3 RMSE: 0.077369
[0]	validation_0-rmse:1.05189
[100]	validation_0-rmse:0.35378
[200]	validation_0-rmse:0.35411
[300]	validation_0-rmse:0.35504
[400]	validation_0-rmse:0.35561
[499]	validation_0-rmse:0.35597
Fold 4 RMSE: 0.355974
[0]	validation_0-rmse:1.54512
[100]	validation_0-rmse:0.19924
[200]	validation_0-rmse:0.20291
[300]	validation_0-rmse:0.20323
[400]	validation_0-rmse:0.20377
[499]	validation_0-rmse:0.20410
Fold 5 RMSE: 0.204102
Mean CV RMSE: 0.219558


# 6. Hyperparameter Tuning (Optuna)
Tune XGBoost hyperparameters with 50 Optuna trials using time-series CV and GPU execution.

In [6]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda"
    }

    cv = TimeSeriesSplit(n_splits=3)
    rmses = []
    for tr_idx, val_idx in cv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
        pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        rmses.append(rmse)

    print(f"Trial {trial.number} | RMSE: {float(np.mean(rmses)):.4f} | params: {trial.params}")
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Best RMSE:", study.best_value)
print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

[I 2026-05-27 19:43:06,419] A new study created in memory with name: no-name-ce35c007-b2ef-434b-9c3c-8a8710fba2a9


[0]	validation_0-rmse:1.05719
[100]	validation_0-rmse:0.40605
[200]	validation_0-rmse:0.39634
[300]	validation_0-rmse:0.39558
[400]	validation_0-rmse:0.39525
[500]	validation_0-rmse:0.39522
[583]	validation_0-rmse:0.39524
[0]	validation_0-rmse:0.70865
[100]	validation_0-rmse:0.08786
[200]	validation_0-rmse:0.09567
[300]	validation_0-rmse:0.09828
[400]	validation_0-rmse:0.09991
[500]	validation_0-rmse:0.10023
[583]	validation_0-rmse:0.10044
[0]	validation_0-rmse:1.41010
[100]	validation_0-rmse:0.64987
[200]	validation_0-rmse:0.65506
[300]	validation_0-rmse:0.65454
[400]	validation_0-rmse:0.65389
[500]	validation_0-rmse:0.65314
[583]	validation_0-rmse:0.65183
Trial 0 | RMSE: 0.3825 | params: {'n_estimators': 584, 'learning_rate': 0.192193458158131, 'max_depth': 4, 'subsample': 0.9341928660507848, 'colsample_bytree': 0.6732190819730731, 'min_child_weight': 5}


[I 2026-05-27 19:43:09,476] Trial 0 finished with value: 0.38250316980618543 and parameters: {'n_estimators': 584, 'learning_rate': 0.192193458158131, 'max_depth': 4, 'subsample': 0.9341928660507848, 'colsample_bytree': 0.6732190819730731, 'min_child_weight': 5}. Best is trial 0 with value: 0.38250316980618543.


[0]	validation_0-rmse:1.15088
[100]	validation_0-rmse:0.42112
[200]	validation_0-rmse:0.40581
[300]	validation_0-rmse:0.40314
[400]	validation_0-rmse:0.39653
[500]	validation_0-rmse:0.39346
[600]	validation_0-rmse:0.39074
[700]	validation_0-rmse:0.38792
[788]	validation_0-rmse:0.38692
[0]	validation_0-rmse:0.81839
[100]	validation_0-rmse:0.07002
[200]	validation_0-rmse:0.07651
[300]	validation_0-rmse:0.07981
[400]	validation_0-rmse:0.08167
[500]	validation_0-rmse:0.08414
[600]	validation_0-rmse:0.08590
[700]	validation_0-rmse:0.08737
[788]	validation_0-rmse:0.08803
[0]	validation_0-rmse:1.52063
[100]	validation_0-rmse:0.65466
[200]	validation_0-rmse:0.64142
[300]	validation_0-rmse:0.63466
[400]	validation_0-rmse:0.62832
[500]	validation_0-rmse:0.62732
[600]	validation_0-rmse:0.62769
[700]	validation_0-rmse:0.62871
[788]	validation_0-rmse:0.63195
Trial 1 | RMSE: 0.3690 | params: {'n_estimators': 789, 'learning_rate': 0.05865178893595365, 'max_depth': 3, 'subsample': 0.8519243160149248, 

[I 2026-05-27 19:43:13,333] Trial 1 finished with value: 0.36896875497227244 and parameters: {'n_estimators': 789, 'learning_rate': 0.05865178893595365, 'max_depth': 3, 'subsample': 0.8519243160149248, 'colsample_bytree': 0.8854620495846149, 'min_child_weight': 4}. Best is trial 1 with value: 0.36896875497227244.


...

Trial 17 | RMSE: 0.3764 | params: {'n_estimators': 853, 'learning_rate': 0.16451196373002197, 'max_depth': 4, 'subsample': 0.8864570400967988, 'colsample_bytree': 0.8521809109824734, 'min_child_weight': 5}
[0]	validation_0-rmse:1.15714
[100]	validation_0-rmse:0.43493
[200]	validation_0-rmse:0.42719
[300]	validation_0-rmse:0.42513
[400]	validation_0-rmse:0.42462
[500]	validation_0-rmse:0.42431
[600]	validation_0-rmse:0.42420
[616]	validation_0-rmse:0.42420
[0]	validation_0-rmse:0.82491
[100]	validation_0-rmse:0.07774
[200]	validation_0-rmse:0.07949
[300]	validation_0-rmse:0.08063
[400]	validation_0-rmse:0.08128
[500]	validation_0-rmse:0.08158
[600]	validation_0-rmse:0.08172
[616]	validation_0-rmse:0.08174
[0]	validation_0-rmse:1.52876
[100]	validation_0-rmse:0.66954
[200]	validation_0-rmse:0.65566
[300]	validation_0-rmse:0.65298
[400]	validation_0-rmse:0.65088
[500]	validation_0-rmse:0.65035
[600]	validation_0-rmse:0.65020
[616]	validation_0-rmse:0.65038


[I 2026-05-27 19:44:13,239] Trial 18 finished with value: 0.385438010014687 and parameters: {'n_estimators': 617, 'learning_rate': 0.04968796623731133, 'max_depth': 7, 'subsample': 0.8381543350521401, 'colsample_bytree': 0.7312365194610061, 'min_child_weight': 3}. Best is trial 2 with value: 0.36832703246570003.


Trial 18 | RMSE: 0.3854 | params: {'n_estimators': 617, 'learning_rate': 0.04968796623731133, 'max_depth': 7, 'subsample': 0.8381543350521401, 'colsample_bytree': 0.7312365194610061, 'min_child_weight': 3}
[0]	validation_0-rmse:1.12799
[100]	validation_0-rmse:0.42518
[200]	validation_0-rmse:0.41471
[295]	validation_0-rmse:0.40966
[0]	validation_0-rmse:0.78994
[100]	validation_0-rmse:0.07449
[200]	validation_0-rmse:0.07988
[295]	validation_0-rmse:0.08293
[0]	validation_0-rmse:1.49684
[100]	validation_0-rmse:0.63998
[200]	validation_0-rmse:0.63248
[295]	validation_0-rmse:0.62786


[I 2026-05-27 19:44:14,594] Trial 19 finished with value: 0.37348608401012334 and parameters: {'n_estimators': 296, 'learning_rate': 0.0939861189047976, 'max_depth': 3, 'subsample': 0.658593442225544, 'colsample_bytree': 0.9254119836001473, 'min_child_weight': 3}. Best is trial 2 with value: 0.36832703246570003.


Trial 19 | RMSE: 0.3735 | params: {'n_estimators': 296, 'learning_rate': 0.0939861189047976, 'max_depth': 3, 'subsample': 0.658593442225544, 'colsample_bytree': 0.9254119836001473, 'min_child_weight': 3}
Best RMSE: 0.36832703246570003
Best params:
  n_estimators: 365
  learning_rate: 0.10905378537177
  max_depth: 4
  subsample: 0.8435452683129303
  colsample_bytree: 0.694880830841114
  min_child_weight: 3


# 7. Final Model Training
Train the final model with best Optuna parameters on the full training set and evaluate on the holdout test set.

In [7]:
final_params = dict(best_params)
final_params.update({
    "random_state": 42,
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "device": "cuda"
})

final_model = XGBRegressor(**final_params)
final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

test_pred = final_model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
mae = float(mean_absolute_error(y_test, test_pred))
mape = float(np.mean(np.abs((y_test - test_pred) / y_test.replace(0, np.nan))) * 100)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.4f}%")

print(f"Retraining on full dataset...")
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])
final_model.fit(X_full, y_full, verbose=100)
print("Final model trained on full dataset")



# AUTO SAVE immediately after training
from pathlib import Path
import joblib, json
_save_dir = Path("./")
joblib.dump(final_model, _save_dir / "xgboost_model.pkl")
with open(_save_dir / "xgboost_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)
with open(_save_dir / "feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)
print("AUTO SAVED")

[0]	validation_0-rmse:1.10946
[100]	validation_0-rmse:0.08932
[200]	validation_0-rmse:0.08991
[300]	validation_0-rmse:0.09213
[364]	validation_0-rmse:0.09242
RMSE: 0.092417
MAE:  0.070659
MAPE: 15.9118%
Retraining on full dataset...
Final model trained on full dataset
AUTO SAVED


In [8]:
import importlib
import nbformat
importlib.reload(nbformat)
print(nbformat.__version__)

5.10.4


# 8. Feature Importance
Visualize the top 15 most important predictors from the trained XGBoost model.

In [9]:
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig_imp = go.Figure(
    go.Bar(
        x=importance.values[::-1],
        y=importance.index[::-1],
        orientation="h",
        marker_color="#2563eb"
    )
)
fig_imp.update_layout(
    title=f"{TICKER} XGBoost Top 15 Feature Importance",
    template="plotly_white",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=600
)
fig_imp.show()

# 9. Forecast Plot
Plot holdout-period actual vs predicted values and a 30-day future forecast using the latest feature row as a simple forward scenario.

In [10]:
future_steps = 30
last_row = X_test.iloc[[-1]] if len(X_test) > 0 else X_train.iloc[[-1]]
future_X = pd.concat([last_row] * future_steps, ignore_index=True)
future_pred = final_model.predict(future_X)

if isinstance(test_df.index, pd.DatetimeIndex):
    future_idx = pd.bdate_range(start=test_df.index[-1] + pd.tseries.offsets.BDay(1), periods=future_steps)
else:
    future_idx = pd.RangeIndex(start=len(test_df), stop=len(test_df) + future_steps)

y_actual = df.loc[test_df.index, "Close"]
pred_actual = preprocessor.inverse_transform(test_pred)
future_pred_actual = preprocessor.inverse_transform(future_pred)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=test_df.index, y=y_actual, mode="lines", name="Actual", line=dict(color="#111827", width=2)))
fig_fc.add_trace(go.Scatter(x=test_df.index, y=pred_actual, mode="lines", name="Predicted", line=dict(color="#2563eb", width=2)))
fig_fc.add_trace(go.Scatter(x=future_idx, y=future_pred_actual, mode="lines", name="30-Day Forecast", line=dict(color="#10b981", width=2, dash="dash")))
fig_fc.update_layout(
    title=f"{TICKER} Actual vs Predicted + 30-Day Forecast",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Close Price (₹)",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_xgboost_forecast.html"
fig_fc.write_html(str(out_path))
print(f"Wrote forecast plot to {out_path.resolve()}")

Wrote forecast plot to C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\RELIANCE_NS_xgboost_forecast.html


# 10. Save Artifacts
Persist the trained model, preprocessing artifacts, feature columns, and best hyperparameters under the ticker-specific artifacts directory.

In [11]:
model_path = ARTIFACTS_DIR / "xgboost_model.pkl"
feature_cols_path = ARTIFACTS_DIR / "feature_columns.json"
best_params_path = ARTIFACTS_DIR / "xgboost_best_params.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print("Saved artifacts:")
print(f"- Model: {model_path.resolve()}")
print(f"- Preprocessor scaler: {(ARTIFACTS_DIR / 'scaler.pkl').resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Saved artifacts:
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\xgboost_model.pkl
- Preprocessor scaler: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\scaler.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\xgboost\xgboost_best_params.json
